# 🏛️ Rajat Alpha V67: Institutional Weekly Swing Trading Screener
### **Engineered for High-Conviction Stage 2 Leaders, Buy Zone Pivots & Risk-Optimized Watchlists**

---

### **Methodology & Institutional Setup Architecture**
This screener implements an institutional quantitative screening system designed to scan the entire US equity universe (S&P 500, Nasdaq 100, Dow Jones 30, and high-liquidity leaders) and output a ranked watchlist in Excel for next week's trading.

#### **1. TradingView Screener Rules (13 Core Gating Filters)**
1. **Price $\ge \$10.00** (Eliminates penny stocks and micro-cap manipulation)
2. **Market Cap $\ge \$2.0B** (Mid-cap and large-cap institutional threshold)
3. **Performance YTD $> +8.0\%$** (Guarantees positive alpha relative to cash/market)
4. **EMA 21 $\ge$ SMA 50** (Intermediate bullish momentum alignment)
5. **Price $\ge$ 52-Week Low $+ 15\%$** (Minervini Rule #6: Out of base / Stage 2 uptrend)
6. **SMA 50 below Price by $0\%$ to $10\%$** (Sweet spot: In accumulation zone near 50 SMA, not overbought)
7. **SMA 200 below Price by $\ge 10\%$** (Macro Stage 2 bull clearance)
8. **SMA 50 $\ge$ SMA 200** (Classic Golden Cross structural alignment)
9. **Volume Change $> +10\%$** (Institutional volume expansion on reversal/pivot)
10. **EMA 10 $\ge$ EMA 21** (Short-term momentum curling upward)
11. **Average Volume (100 days) $> 250,000$ shares** (Institutional execution liquidity)
12. **SMA 150 $\ge$ SMA 200** (Minervini Trend Template Rule #4)
13. **SMA 50 $\ge$ SMA 150** (Minervini Trend Template Rule #5: Full $50 > 150 > 200$ moving average fan)

#### **2. Senior Strategist Alpha Upgrades**
- **Minervini 52-Week High Rule**: Price within $25\%$ of 52-week High (eliminates stocks fighting overhead resistance).
- **Moving Average Curvature & Slopes**: Verifies that 21 EMA is upward ($>0$) and 50 SMA is upward/flat ($\ge 0$).
- **Precision Buy Zone Classification**: Identifies if the stock is at the **21 EMA Prime Zone** ($\le 2.5\%$), **50 SMA Demand Zone** ($\le 3.5\%$), or **Dual Convergence Sweet Spot**.
- **30-Day Swing Pivot Structure**: Verifies **Higher Swing Lows ($SL_2 \ge SL_1$)** and Volatility Contraction Pattern (VCP).
- **Institutional Quality Score (IQS: 0–100) & Star Rating (⭐⭐⭐⭐⭐)**: Deterministic ranking prioritizing Grade A+ setups.
- **Automated Trade Plan**: Computes Entry, ATR Stop Loss (bounded 5–10%), Target 1 (+5%), Target 2 (+10%), Target 3 (+15%), and Risk/Reward ratio.

---


In [ ]:
# @title 📦 Step 1: Install Dependencies & Import Modules
!pip install --quiet yfinance openpyxl matplotlib pandas numpy tqdm

import os
import sys
import re
import urllib.request
import ssl
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any

import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from tqdm.notebook import tqdm

print('✅ Environment Initialized. Ready to scan!')


In [ ]:
# @title ⚙️ Step 2: Configure Screener Parameters
# @markdown Select index universe and export parameters:

universe_choice = "combined" #@param ["combined", "sp500", "nasdaq100", "dow30", "liquid_leaders"]
output_excel_name = "weekly_swing_screener_results.xlsx" #@param {type:"string"}
max_watchlist_symbols = 25 #@param {type:"integer"}
require_higher_swing_lows = True #@param {type:"boolean"}
within_25pct_of_52w_high = True #@param {type:"boolean"}

print(f'Selected Universe: {universe_choice.upper()}')
print(f'Output Excel File: {output_excel_name}')
print(f'Watchlist Target Size: {max_watchlist_symbols} setups')


In [ ]:
# @title 🧠 Step 3: Define Quantitative Screener Engine & Indicators

DOW_30 = [
    "AAPL", "AMGN", "AMZN", "AXP", "BA", "CAT", "CRM", "CSCO", "CVX", "DIS",
    "DOW", "GS", "HD", "HON", "IBM", "INTC", "JNJ", "JPM", "KO", "MCD",
    "MMM", "MRK", "MSFT", "NKE", "NVDA", "PG", "SHW", "TRV", "UNH", "V", "WMT"
]

NASDAQ_100 = [
    "AAPL", "ABNB", "ADBE", "ADI", "ADP", "ADSK", "AEP", "AMAT", "AMD", "AMGN",
    "AMZN", "ANSS", "APP", "ARM", "ASML", "AVGO", "AXON", "BIIB", "BKNG", "BKR",
    "CCEP", "CDNS", "CDW", "CEG", "CHTR", "CPRT", "CRWD", "CSCO", "CSX", "CTAS",
    "CTSH", "DASH", "DDOG", "DLTR", "DXCM", "EA", "EXC", "FANG", "FAST", "FTNT",
    "GEHC", "GFS", "GILD", "GOOG", "GOOGL", "HON", "IDXX", "ILMN", "INTC", "INTU",
    "ISRG", "KDP", "KHC", "KLAC", "LIN", "LRCX", "LULU", "MAR", "MCHP", "MDB",
    "MDLZ", "MELI", "META", "MNST", "MRNA", "MRVL", "MSFT", "MU", "NFLX", "NVDA",
    "NXPI", "ODFL", "ON", "ORLY", "PANW", "PAYX", "PCAR", "PDD", "PEP", "PLTR",
    "PYPL", "QCOM", "REGN", "ROP", "ROST", "SBUX", "SNPS", "TEAM", "TMUS", "TSLA",
    "TTD", "TXN", "VRSK", "VRTX", "WBD", "WDAY", "XEL", "ZS"
]

SP_500_SAMPLE = [
    "AAPL", "NVDA", "MSFT", "AMZN", "META", "GOOGL", "GOOG", "BRK-B", "LLY", "AVGO",
    "TSLA", "JPM", "WMT", "UNH", "XOM", "V", "PG", "MA", "COST", "JNJ",
    "ORCL", "HD", "BAC", "ABBV", "CVX", "NFLX", "KO", "MRK", "CRM", "AMD",
    "PEP", "LIN", "TMO", "ACN", "WFC", "CSCO", "MCD", "ABT", "GE", "IBM",
    "PM", "ISRG", "TXN", "CAT", "NOW", "QCOM", "VZ", "MS", "AMAT", "UBER",
    "GS", "SPGI", "AXP", "BKNG", "RTX", "HON", "LOW", "PFE", "SYK", "UNP",
    "BLK", "TJX", "VRTX", "REGN", "ETN", "PANW", "PGR", "LRCX", "CB", "BSX",
    "ADI", "CI", "C", "MDT", "FI", "PLTR", "ANET", "KLAC", "SCHW", "SNPS",
    "CDNS", "CRWD", "MAR", "APH", "EOG", "TDG", "WM", "ITW", "SHW", "CTAS",
    "FTNT", "ORLY", "COF", "MCK", "NXPI", "DE", "WELL", "PH", "EMR", "PCAR"
]

def get_universe_symbols(u_choice: str) -> List[str]:
    syms = set()
    if u_choice in ["sp500", "combined"]:
        try:
            ctx = ssl._create_unverified_context()
            req = urllib.request.Request("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, context=ctx, timeout=6) as resp:
                html = resp.read().decode('utf-8')
            raw = re.findall(r'href="https://www\.nyse\.com/quote/XNYS:([A-Z\.]+)"', html)
            if not raw:
                raw = re.findall(r'<td><a rel="nofollow" class="external text" href="[^"]+">([A-Z\.\-]+)</a></td>', html)
            syms.update([s.replace('.', '-') for s in raw if len(s) <= 5])
        except Exception:
            syms.update(SP_500_SAMPLE)
    if u_choice in ["nasdaq100", "combined"]:
        syms.update(NASDAQ_100)
    if u_choice in ["dow30", "combined"]:
        syms.update(DOW_30)
    if u_choice == "liquid_leaders" or not syms:
        syms.update(SP_500_SAMPLE + NASDAQ_100 + DOW_30)
    return sorted(list(syms))

def calculate_screener_indicators(df: pd.DataFrame) -> Optional[pd.DataFrame]:
    if df is None or len(df) < 200:
        return None
    df = df.copy()
    c, h, l, v = df['Close'], df['High'], df['Low'], df['Volume']
    
    df['EMA10'] = c.ewm(span=10, adjust=False).mean()
    df['EMA21'] = c.ewm(span=21, adjust=False).mean()
    df['SMA50'] = c.rolling(50).mean()
    df['SMA150'] = c.rolling(150).mean()
    df['SMA200'] = c.rolling(200).mean()
    df['VOL_SMA21'] = v.rolling(21).mean()
    df['VOL_SMA100'] = v.rolling(100).mean()
    df['HIGH_52W'] = h.rolling(min(252, len(df))).max()
    df['LOW_52W'] = l.rolling(min(252, len(df))).min()
    
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    df['ATR14'] = tr.rolling(14).mean()
    
    df['SLOPE_EMA21'] = ((df['EMA21'] - df['EMA21'].shift(5)) / df['EMA21'].shift(5)) * 100.0
    df['SLOPE_SMA50'] = ((df['SMA50'] - df['SMA50'].shift(10)) / df['SMA50'].shift(10)) * 100.0
    df['SLOPE_SMA200'] = ((df['SMA200'] - df['SMA200'].shift(20)) / df['SMA200'].shift(20)) * 100.0
    return df

def analyze_swing_structure(df: pd.DataFrame, window_bars: int = 30) -> Dict[str, Any]:
    if len(df) < window_bars + 2:
        return {'valid_structure': False, 'higher_low': False, 'contraction': False}
    sub = df.iloc[-(window_bars + 2):]
    highs, lows = [], []
    for i in range(1, len(sub) - 1):
        if sub['High'].iloc[i] > sub['High'].iloc[i-1] and sub['High'].iloc[i] > sub['High'].iloc[i+1]:
            highs.append(float(sub['High'].iloc[i]))
        if sub['Low'].iloc[i] < sub['Low'].iloc[i-1] and sub['Low'].iloc[i] < sub['Low'].iloc[i+1]:
            lows.append(float(sub['Low'].iloc[i]))
    has_swings = len(highs) >= 2 and len(lows) >= 2
    higher_low = False
    contraction = False
    if len(lows) >= 2:
        higher_low = lows[-1] >= (lows[-2] * 0.995)
    if has_swings:
        contraction = (highs[-1] - lows[-1]) <= ((highs[-2] - lows[-2]) * 1.05)
    return {'valid_structure': has_swings, 'higher_low': higher_low, 'contraction': contraction, 'sl1': lows[-2] if len(lows)>=2 else 0, 'sl2': lows[-1] if len(lows)>=2 else 0}

def evaluate_candidate(symbol: str, df: pd.DataFrame, spy_df: Optional[pd.DataFrame] = None) -> Optional[Dict[str, Any]]:
    if df is None or len(df) < 200: return None
    c_bar, p_bar = df.iloc[-1], df.iloc[-2]
    close, prev_close = float(c_bar['Close']), float(p_bar['Close'])
    volume, prev_volume = float(c_bar['Volume']), float(p_bar['Volume'])
    
    ema10, ema21 = float(c_bar['EMA10']), float(c_bar['EMA21'])
    sma50, sma150, sma200 = float(c_bar['SMA50']), float(c_bar['SMA150']), float(c_bar['SMA200'])
    vol_sma21 = float(c_bar['VOL_SMA21']) if not pd.isna(c_bar['VOL_SMA21']) else volume
    vol_sma100 = float(c_bar['VOL_SMA100']) if not pd.isna(c_bar['VOL_SMA100']) else volume
    high52w, low52w = float(c_bar['HIGH_52W']), float(c_bar['LOW_52W'])
    atr14 = float(c_bar['ATR14']) if not pd.isna(c_bar['ATR14']) else close * 0.03
    
    # --- 13 TRADINGVIEW CORE RULES ---
    if close < 10.0: return None
    if vol_sma100 < 250000: return None
    if ema21 < sma50: return None
    if (close - low52w) / low52w < 0.15: return None
    dist_sma50 = (close - sma50) / sma50
    if dist_sma50 < 0.0 or dist_sma50 > 0.10: return None
    dist_sma200 = (close - sma200) / sma200
    if dist_sma200 < 0.10: return None
    if sma50 < sma200: return None
    if (volume - prev_volume)/prev_volume <= 0.10 and (volume - vol_sma21)/vol_sma21 <= 0.10: return None
    if ema10 < ema21: return None
    if sma150 < sma200: return None
    if sma50 < sma150: return None
    
    current_year = df.index[-1].year
    ytd_bars = df[df.index.year == current_year]
    ytd_perf = (close - float(ytd_bars['Open'].iloc[0])) / float(ytd_bars['Open'].iloc[0]) if len(ytd_bars) >= 2 else (close - float(df['Close'].iloc[-min(126, len(df))])) / float(df['Close'].iloc[-min(126, len(df))])
    if ytd_perf <= 0.08: return None
    
    # --- SENIOR STRATEGIST ALPHA CHECKS ---
    off_52w_high = (high52w - close) / high52w
    if within_25pct_of_52w_high and off_52w_high > 0.25: return None
    
    slope_ema21 = float(c_bar['SLOPE_EMA21']) if not pd.isna(c_bar['SLOPE_EMA21']) else 0.0
    slope_sma50 = float(c_bar['SLOPE_SMA50']) if not pd.isna(c_bar['SLOPE_SMA50']) else 0.0
    dist_ema21_pct = (close - ema21) / ema21 * 100.0
    dist_sma50_pct = (close - sma50) / sma50 * 100.0
    
    if dist_ema21_pct <= 2.5 and dist_sma50_pct <= 3.5:
        buy_zone = "Dual Convergence Sweet Spot"
    elif dist_ema21_pct <= 2.5:
        buy_zone = "21 EMA Prime Zone"
    elif dist_sma50_pct <= 3.5:
        buy_zone = "50 SMA Demand Zone"
    else:
        buy_zone = "Upper Pullback Zone"
        
    swing_info = analyze_swing_structure(df, window_bars=30)
    if require_higher_swing_lows and not swing_info.get('higher_low', False):
        return None
        
    # IBD RS Rating
    rs_score = 50.0
    if spy_df is not None and len(spy_df) >= 200:
        try:
            stk_w = 0.4*(close/df['Close'].iloc[-63]-1) + 0.2*(df['Close'].iloc[-63]/df['Close'].iloc[-126]-1) + 0.2*(df['Close'].iloc[-126]/df['Close'].iloc[-189]-1) + 0.2*(df['Close'].iloc[-189]/df['Close'].iloc[-252]-1)
            spy_w = (spy_df['Close'].iloc[-1]/spy_df['Close'].iloc[-252]) - 1.0
            rs_score = round(max(1.0, min(99.0, 50.0 + (stk_w - spy_w)*150.0)), 1)
        except Exception: rs_score = 50.0
        
    # Institutional Quality Score (IQS: 0-100)
    pts = 0
    if ema10 > ema21 > sma50 > sma150 > sma200: pts += 15
    if slope_ema21 > 0: pts += 8
    if slope_sma50 >= 0: pts += 7
    if buy_zone == "Dual Convergence Sweet Spot": pts += 25
    elif "21 EMA" in buy_zone: pts += 20
    elif "50 SMA" in buy_zone: pts += 18
    else: pts += 10
    if swing_info.get('higher_low'): pts += 12
    if swing_info.get('contraction'): pts += 8
    rvol = volume / vol_sma21 if vol_sma21 > 0 else 1.0
    if rvol >= 1.5: pts += 10
    elif rvol >= 1.1: pts += 7
    if close > prev_close: pts += 5
    if rs_score >= 80: pts += 6
    if off_52w_high <= 0.25: pts += 4
    iqs = min(100, pts)
    
    stars = "⭐⭐⭐⭐⭐ (Grade A+ Elite)" if iqs >= 90 else ("⭐⭐⭐⭐ (Grade A High Conviction)" if iqs >= 80 else "⭐⭐⭐ (Grade B+ Setup)")
    
    # Trade Plan
    entry = round(close, 2)
    sl = round(min(entry * 0.95, max(entry * 0.90, entry - 2.0 * atr14)), 2)
    risk_dlr = entry - sl
    risk_pct = round((risk_dlr / entry) * 100.0, 1)
    t1 = round(entry * 1.05, 2)
    t2 = round(entry * 1.10, 2)
    t3 = round(entry * 1.15, 2)
    rr = round((t2 - entry) / risk_dlr, 2) if risk_dlr > 0 else 0.0
    
    return {
        'symbol': symbol, 'close': close, 'ytd_perf': round(ytd_perf*100, 1), 'rvol': round(rvol, 2),
        'dist_ema21': round(dist_ema21_pct, 1), 'dist_sma50': round(dist_sma50_pct, 1), 'off_52w_high': round(off_52w_high*100, 1),
        'slope_ema21': round(slope_ema21, 2), 'slope_sma50': round(slope_sma50, 2), 'buy_zone': buy_zone,
        'higher_low': 'YES' if swing_info.get('higher_low') else 'NO', 'sl1': round(swing_info.get('sl1', 0), 2), 'sl2': round(swing_info.get('sl2', 0), 2),
        'rs_score': rs_score, 'iqs_score': iqs, 'star_rating': stars, 'entry': entry, 'stop_loss': sl, 'risk_pct': risk_pct,
        't1': t1, 't2': t2, 't3': t3, 'rr': rr
    }


In [ ]:
# @title 🚀 Step 4: Run Institutional Screener Pipeline

tickers = get_universe_symbols(universe_choice)
print(f'Fetching benchmark SPY and {len(tickers)} tickers across {universe_choice.upper()}...')

spy_data = yf.download('SPY', period='2y', interval='1d', progress=False)
if isinstance(spy_data.columns, pd.MultiIndex): spy_data.columns = spy_data.columns.get_level_values(0)
spy_ind = calculate_screener_indicators(spy_data)

results = []
batch_size = 50

for b_start in tqdm(range(0, len(tickers), batch_size), desc='Scanning Tickers'):
    b_syms = tickers[b_start:b_start + batch_size]
    try:
        data = yf.download(b_syms, period='2y', interval='1d', group_by='ticker', progress=False, threads=True)
        for sym in b_syms:
            try:
                df = data[sym].dropna(how='all') if len(b_syms) > 1 else data.dropna(how='all')
                if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
                if len(df) < 200: continue
                ind = calculate_screener_indicators(df)
                if ind is None: continue
                eval_res = evaluate_candidate(sym, ind, spy_df=spy_ind)
                if eval_res is not None:
                    results.append(eval_res)
            except Exception: pass
    except Exception: pass

results.sort(key=lambda x: (x['iqs_score'], x['rs_score'], -abs(x['dist_ema21'])), reverse=True)
print(f'\n🎯 Screening Complete: {len(results)} symbols passed all 13 TradingView & Minervini filters!')

# Display Top Setups in DataFrame
df_res = pd.DataFrame(results[:max_watchlist_symbols])
if not df_res.empty:
    display_cols = ['symbol', 'close', 'iqs_score', 'star_rating', 'buy_zone', 'dist_ema21', 'dist_sma50', 'rs_score', 'entry', 'stop_loss', 'risk_pct', 't2', 'rr']
    display(df_res[display_cols])
else:
    print('No candidates passed strict filters. Try selecting "combined" universe or loosening higher swing low rule.')


In [ ]:
# @title 📊 Step 5: Export to Formatted Multi-Sheet Excel & 1-Click Download

wb = openpyxl.Workbook()
wb.remove(wb.active)

# Styling
navy_fill = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")
gold_fill = PatternFill(start_color="B28900", end_color="B28900", fill_type="solid")
emerald_fill = PatternFill(start_color="E2EFDA", end_color="E2EFDA", fill_type="solid")
h_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
reg_font = Font(name="Calibri", size=10)

# Sheet 1: Top Watchlist
ws1 = wb.create_sheet(title="Top_Watchlist_Next_Week")
headers1 = ["Rank", "Symbol", "IQS Score", "Rating", "Price ($)", "Buy Zone", "Dist 21 EMA (%)", "Dist 50 SMA (%)", "21 EMA Slope (%)", "50 SMA Slope (%)", "Higher Lows", "IBD RS", "Entry ($)", "Stop Loss ($)", "Risk (%)", "PT1 (+5%)", "PT2 (+10%)", "PT3 (+15%)", "R:R"]
ws1.append(headers1)
for c in range(1, len(headers1) + 1):
    cell = ws1.cell(1, c)
    cell.fill = navy_fill; cell.font = h_font; cell.alignment = Alignment(horizontal="center")

top_picks = results[:max_watchlist_symbols]
for idx, r in enumerate(top_picks, 1):
    row = [idx, r['symbol'], r['iqs_score'], r['star_rating'], r['close'], r['buy_zone'], f"{r['dist_ema21']:+.1f}%", f"{r['dist_sma50']:+.1f}%", f"{r['slope_ema21']:+.1f}%", f"{r['slope_sma50']:+.1f}%", r['higher_low'], r['rs_score'], r['entry'], r['stop_loss'], f"{r['risk_pct']:.1f}%", r['t1'], r['t2'], r['t3'], f"1:{r['rr']:.2f}"]
    ws1.append(row)
    r_num = ws1.max_row
    if r['iqs_score'] >= 90:
        ws1.cell(r_num, 2).fill = emerald_fill
        ws1.cell(r_num, 3).fill = emerald_fill

# Sheet 2: All Screened Candidates
ws2 = wb.create_sheet(title="All_Screened_Candidates")
headers2 = ["Symbol", "Price", "YTD Perf (%)", "RVol", "IQS Score", "Rating", "Buy Zone", "Dist EMA21 (%)", "Dist SMA50 (%)", "Off 52w High (%)", "Higher Low", "SL1", "SL2", "RS Rating", "Entry", "Stop Loss", "Risk (%)", "PT1", "PT2", "PT3", "R:R"]
ws2.append(headers2)
for c in range(1, len(headers2) + 1):
    cell = ws2.cell(1, c)
    cell.fill = gold_fill; cell.font = h_font; cell.alignment = Alignment(horizontal="center")
for r in results:
    ws2.append([r['symbol'], r['close'], f"{r['ytd_perf']:+.1f}%", r['rvol'], r['iqs_score'], r['star_rating'], r['buy_zone'], r['dist_ema21'], r['dist_sma50'], r['off_52w_high'], r['higher_low'], r['sl1'], r['sl2'], r['rs_score'], r['entry'], r['stop_loss'], f"{r['risk_pct']:.1f}%", r['t1'], r['t2'], r['t3'], r['rr']])

# Sheet 3: Watchlist Export
ws3 = wb.create_sheet(title="Watchlist_TXT_Export")
ws3.append(["Symbol (Copy to config/watchlist.txt)", "IQS Score", "Rating"])
ws3.cell(1, 1).fill = navy_fill; ws3.cell(1, 1).font = h_font
ws3.cell(1, 2).fill = navy_fill; ws3.cell(1, 2).font = h_font
ws3.cell(1, 3).fill = navy_fill; ws3.cell(1, 3).font = h_font
for r in top_picks:
    ws3.append([r['symbol'], r['iqs_score'], r['star_rating']])

# Auto column width
for ws in [ws1, ws2, ws3]:
    for col in ws.columns:
        max_l = max(len(str(c.value or '')) for c in col)
        ws.column_dimensions[get_column_letter(col[0].column)].width = max(max_l + 3, 12)

wb.save(output_excel_name)
print(f'✅ Excel file successfully created: {output_excel_name}')

# Direct Google Colab Download
try:
    from google.colab import files
    files.download(output_excel_name)
    print('📥 Download dialog triggered in your browser!')
except Exception:
    print(f'File saved locally as: {output_excel_name}')
